# Этап 3: Отслеживание Сноубордиста и Имитация Камеры Дрона

На этом финальном этапе проекта мы интегрируем обученную модель обнаружения объектов с алгоритмом отслеживания для создания полноценной системы слежения за сноубордистом.

**Цель этапа:** Разработать и продемонстрировать систему, способную в реальном времени:

1. Надежно отслеживать одного целевого сноубордиста в видеопотоке, поддерживая его уникальный ID.

2. Центрировать этот целевой объект в кадре, имитируя работу камеры дрона или другого устройства, автоматически следующего за объектом.

3. Генерировать управляющие команды для такого дрона на основе положения и размера отслеживаемого объекта в исходном кадре.

**Мотивация для отслеживания**
Одного лишь обнаружения объектов (как на Этапе 2) недостаточно для большинства интерактивных приложений. Для таких задач, как следование дрона, анализ движения или подсчет уникальных объектов, необходимо **отслеживать** объекты, то есть поддерживать их идентичность через последовательность кадров. Это позволяет нам не только знать, где находится объект, но и какой это объект, а также как он движется со временем.

Мы используем подход **Tracking-by-Detection (отслеживание на основе детекции)**, который является наиболее распространенным и эффективным для задач в реальном времени. В рамках этого подхода, детектор объектов (наша обученная YOLO11n) сначала находит все интересующие объекты в каждом кадре, а затем алгоритм трекинга связывает эти детекции с существующими или новыми траекториями.

**Ключевые особенности этого этапа:**
* **Интеграция YOLO11n с трекером:** Объединение нашей высокопроизводительной модели-детектора с надежным алгоритмом отслеживания.

* **Динамическое центрирование объекта:** Реализация логики автоматической обрезки видеокадра вокруг целевого сноубордиста, создавая эффект "следующей камеры".

* **Генерация команд дрона:** Преобразование пространственных координат и размера объекта в понятные команды для автономного управления дроном (движение по горизонтали, вертикали, изменение дистанции).

* **Логирование данных:** Детальная запись всех ключевых параметров отслеживания и команд дрона в структурированный файл для последующего анализа и визуализации.

* **Комплексное демонстрационное видео:** Создание наглядного видео с сравнением исходного и обработанного кадров, а также информативным оверлеем, демонстрирующим метрики и команды в реальном времени.

### 1. Подготовка Окружения и Импорты

Начнем с настройки рабочего окружения и импорта всех необходимых библиотек и функций.

In [ ]:
# --- Импорты ---
import os
import sys
import yaml

# Добавление корневой директории проекта в sys.path, позволяет импортировать модули из папок, расположенных на одном уровне с 'notebooks/', например из 'src/'
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

# Импортируем функцию из скрипта для отслеживания и утилиту для генерации имен запусков
from src.tracker import track_video_and_center_object
from src.utils import get_next_run_name
from src.visualization_utils import create_side_by_side_demo_video

### 2. Конфигурация Пайплайна Отслеживания

Прежде чем начать процесс отслеживания, необходимо определить пути к видеофайлам и обученной модели, а также настроить ключевые параметры для детекции и трекинга.

In [2]:
# Загрузка конфигурации из config.yaml
with open('../config.yaml', 'r') as file:
    config = yaml.safe_load(file)

# Путь к исходному видео и модели из config
video_input_path = config['paths']['video_input_path']
model_path = config['paths']['model_path']

# Определение имени для текущего запуска отслеживания
track_run_name = get_next_run_name("snowboarder_tracking", runs_relative_path=os.path.join(project_root, 'runs', 'track'))
video_output_path = config['paths']['video_output_path'].format(track_run_name=track_run_name)
tracking_log_path = os.path.join(os.path.dirname(video_output_path), "tracking_log.jsonl")
demo_output_path = os.path.join(os.path.dirname(video_output_path), "snowboarder_tracking_demo.mp4")                                                                    

print("Настроены пути и параметры для текущего запуска:")
print(f"   Исходное видео: {video_input_path}")
print(f"   Модель для отслеживания: {model_path}")
print(f"   Выходное видео (отслеживание) будет сохранено в: {video_output_path}")
print(f"   Файл логов трекинга: {tracking_log_path}")
print(f"   Финальное демо-видео будет сохранено в: {demo_output_path}")
print(f"   Имя текущего запуска отслеживания: {track_run_name}")

Настроены пути и параметры для текущего запуска:
   Исходное видео: ../resources/snowboard_quick_test.mp4
   Модель для отслеживания: ../runs/detect/yolo11n_snowboarder_detection_v1/weights/best.pt
   Выходное видео (отслеживание) будет сохранено в: ../runs/track/snowboarder_tracking_v6/tracked_snowboarder.mp4
   Файл логов трекинга: ../runs/track/snowboarder_tracking_v6\tracking_log.jsonl
   Финальное демо-видео будет сохранено в: ../runs/track/snowboarder_tracking_v6\snowboarder_tracking_demo.mp4
   Имя текущего запуска отслеживания: snowboarder_tracking_v6


### 3. Выполнение Отслеживания и Центрирования
Запуск основной функции `track_video_and_center_object` из скрипт отслеживания `src/tracker.py` для обработки видео.
Этот скрипт выполняет следующие ключевые операции:
1.  **Загружает обученную модель YOLO11n**, полученную на предыдущем этапе.
2.  **Покадрово обрабатывает исходное видео**, выполняя детекцию сноубордистов.
3.  **Применяет алгоритм отслеживания ByteTrack** для поддержания уникальных ID объектов.
4.  **Определяет целевого сноубордиста** (наибольший объект в первом кадре, затем отслеживает его по ID).
5.  **Вычисляет команды для дрона** на основе положения и размера целевого сноубордиста **в исходном кадре** (а не в центрированном).
6.  **Создает новое видео**, где целевой сноубордист **центрирован** в кадре путем обрезки и добавления черных полей (padding).
7.  **Сохраняет все важные данные** о статусе отслеживания, уверенности модели, относительном размере объекта и сгенерированных командах дрона в файл `tracking_log.jsonl`. Эти данные будут использованы для создания информативного оверлея в финальном демонстрационном видео.

In [3]:
print("\nНачинаем процесс отслеживания и центрирования объекта...")

track_video_and_center_object(
    model_path=model_path,
    video_input_path=video_input_path,
    video_output_path=video_output_path,
    config=config
)

print("Процесс отслеживания завершен.")
print(f"Результаты можно найти в папке: {os.path.dirname(video_output_path)}/")


Начинаем процесс отслеживания и центрирования объекта...
Модель успешно загружена из: '../runs/detect/yolo11n_snowboarder_detection_v1/weights/best.pt'
Выходная директория для видео и логов: '../runs/track/snowboarder_tracking_v6'
Исходное видео: '../runs/track/snowboarder_tracking_v6/tracked_snowboarder.mp4'
Разрешение: 1280x720, FPS: 30.00, Всего кадров: 2313
Попытка инициализации VideoWriter с кодеком 'mp4v' и расширением '.mp4'...
Успешно инициализирован VideoWriter с кодеком 'mp4v'.
Данные трекинга будут записаны в: '../runs/track/snowboarder_tracking_v6\tracking_log.jsonl'.
--- Обработано кадров: 1/2313 (0.0%) ---
Кадр 1: Инициализация трекинга. Выбран целевой сноубордист с ID 1.
--- Обработано кадров: 2/2313 (0.1%) ---
--- Обработано кадров: 3/2313 (0.1%) ---
--- Обработано кадров: 4/2313 (0.2%) ---
--- Обработано кадров: 5/2313 (0.2%) ---
--- Обработано кадров: 6/2313 (0.3%) ---
--- Обработано кадров: 7/2313 (0.3%) ---
--- Обработано кадров: 8/2313 (0.3%) ---
--- Обработано ка

### 4. Создание демо-видео

Последний шаг — это объединение всех результатов в одно наглядное демонстрационное видео. Для этого используется функция `create_side_by_side_demo_video` из `src/visualization_utils.py`. Это видео служит основной демонстрацией функциональности всего проекта.

Видео будет скомпоновано следующим образом:
* **Верхняя секция:** Разделена на две части:
    * **Левая часть (60% ширины):** Исходное видео, показывающее полную картину сцены.
    * **Правая часть (40% ширины):** Обработанное видео, где целевой сноубордист центрирован в кадре и обведен ограничивающей рамкой с ID.
* **Нижняя секция:** Специализированная область оверлея, содержащая ключевую информацию о процессе отслеживания и сгенерированных командах дрона.

### **Информация в оверлее:**

В нижней части демонстрационного видео будут выводиться следующие данные, считанные из файла `tracking_log.jsonl`:
* **Статус отслеживания:** Текущее состояние трекера (например, "Отслеживание ID: X", "Поиск объекта...", "Потерян").
* **Уверенность детекции:** Насколько модель уверена в текущем обнаружении целевого объекта.
* **Относительный размер объекта:** Показывает, насколько текущий размер ограничивающей рамки объекта соответствует целевому размеру в кадре (который определяет команду "Дистанция" для дрона). Сопровождается текстовым пояснением (например, "(Приблизься!)", "(Отдалиться!)", "(ОК)").
* **Команды дрона:** Генерируемые команды для горизонтального, вертикального движения и изменения дистанции, которые показывают, как дрон должен корректировать свою позицию, чтобы удерживать сноубордиста в центре и на оптимальном расстоянии. Эти команды рассчитываются на основе положения объекта **в исходном видеокадре**.

**Примечание:** Для корректного отображения русского текста в оверлее используется библиотека `Pillow` и шрифт `arial.ttf`, который должен быть расположен в папке `resources/`.

In [4]:
print(f"\nНачинаем создание демонстрационного видео: {demo_output_path}")
create_side_by_side_demo_video(
    original_video_path=video_input_path, # Путь к исходному видео
    tracked_video_path=video_output_path, # Путь к обработанному видео
    tracking_log_path=tracking_log_path,  # Путь к файлу логов
    output_demo_path=demo_output_path,
    tracked_video_fixed_size=640, # Размер квадратного видео
    overlay_height=120 # Высота оверлея
)

print("Создание демонстрационного видео завершено.")


Начинаем создание демонстрационного видео: ../runs/track/snowboarder_tracking_v6\snowboarder_tracking_demo.mp4
Успешно загружены данные трекинга из: '../runs/track/snowboarder_tracking_v6\tracking_log.jsonl'. Всего записей: 2313.
Создана/проверена выходная директория для демо-видео: '../runs/track/snowboarder_tracking_v6'
Попытка инициализации VideoWriter для демо-видео с кодеком 'mp4v' и расширением '.mp4'...
Успешно инициализирован VideoWriter для демо-видео с кодеком 'mp4v'. Видео будет сохранено как: '../runs/track/snowboarder_tracking_v6\snowboarder_tracking_demo.mp4'.
Создание демо-видео: '../runs/track/snowboarder_tracking_v6\snowboarder_tracking_demo.mp4' с разрешением 1600x760.
Создано демо-кадров: 100
Создано демо-кадров: 200
Создано демо-кадров: 300
Создано демо-кадров: 400
Создано демо-кадров: 500
Создано демо-кадров: 600
Создано демо-кадров: 700
Создано демо-кадров: 800
Создано демо-кадров: 900
Создано демо-кадров: 1000
Создано демо-кадров: 1100
Создано демо-кадров: 1200


### 5. Завершение этапа и выводы

На этом этапе мы успешно реализовали и протестировали полную систему отслеживания и центрирования сноубордиста. Интеграция обученной модели YOLO11n с алгоритмом ByteTrack и логикой генерации команд для дрона продемонстрировала:

* Надежное отслеживание: Модель эффективно поддерживает уникальный ID целевого сноубордиста на протяжении видео.

* Автоматическое центрирование: Видеокадр динамически подстраивается, чтобы удерживать сноубордиста в центре.

* Практические команды: Генерируемые команды дрона показывают применимость системы для задач автономного следования.

Все результаты, включая отслеженное видео, подробные логи и финальное демонстрационное видео, сохранены в структурированной директории `runs/track/`.

Этот этап является кульминацией проекта, демонстрируя сквозной пайплайн от подготовки данных до получения прикладного результата.
Для полного обзора проекта, его целей, архитектуры, результатов и будущих улучшений, пожалуйста, ознакомьтесь с файлом `README.md` в корневом каталоге проекта.